In [30]:
from AgentsClient import HelloAgentsLLM
import ast

In [31]:
PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。
你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每一个步骤都是独立的、可执行的子任务，并且严格按照逻辑顺序排列
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {user_question}

请严格按照以下格式输出你的计划，```python与```作为前后缀:
```python
["步骤1", "步骤2", "步骤3", ...]
```

"""

In [32]:
class Planner:
    def __init__(self, llm_client: HelloAgentsLLM):
        self.llm_client = llm_client
    
    def plan(self, user_question):
        prompt = PLANNER_PROMPT_TEMPLATE.format(user_question=user_question)
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages)
        print(f"计划已生成: {response_text}")

        # 解析计划文本，提取步骤
        try:
            # 找到python块中的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            plan = ast.literal_eval(plan_str) # 将字符串形式的 Python 字面量转换为真正的 Python 对象
            return plan if isinstance(plan, list) else [] # 判断一个对象是否属于某个类型（type）或某些类型
        except (SyntaxError, ValueError, IndexError) as e:
            print(f"计划解析错误: {e}")
            return []
        except Exception as e:
            print(f"计划解析未知错误: {e}")
            return []


In [33]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。
你的任务是严格按照给定的计划， 一步步地解决问题。
你将收到原始问题、完整计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决”当前步骤“，并输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{original_question}

# 完整计划:
{full_plan}

# 到目前为止已经完成的步骤和结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对”当前步骤“的回答:
"""

In [34]:
class Executor:
    def __init__(self, llm_client: HelloAgentsLLM):
        self.llm_client = llm_client
    
    def execute(self, user_question: str, plan: list[str]) -> str:
        """
        执行计划中的每个步骤，返回最终结果。
        """
        history = "" # 存储历史步骤和结果
        
        for i, step in enumerate(plan):
            print(f"执行步骤{i+1}/{len(plan)}")
        
            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                original_question=user_question,
                full_plan=plan,
                history=history if history else "无",
                current_step=step
            )

            messages = [
                {"role": "user", "content": prompt}
            ]

            response_text = self.llm_client.think(messages=messages) or ""

            history += f"步骤{i+1}: {step}\n结果: {response_text}\n"

            print(f"步骤{i+1}结果: {response_text}")

        final_answer = response_text
        return final_answer

In [35]:
class PlanAndSolveAgent:
    def __init__(self, llm_client: HelloAgentsLLM):
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)
    
    def run(self, user_question: str):
        """
        运行计划和解决代理，返回最终结果。
        """
        plan = self.planner.plan(user_question)


        print("="*50)
        print(f"生成的计划: {plan}")
        print("="*50)
        
        if not plan:
            print("计划生成失败")
            return

        final_answer = self.executor.execute(user_question, plan)
        print("="*50)
        print(f"最终答案: {final_answer}")
        print("="*50)

if __name__ == "__main__":
    llm_client = HelloAgentsLLM()
    plan_solve_agent = PlanAndSolveAgent(llm_client)
    plan_solve_agent.run("一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？")


正在调用qwen-plus模型...
大语言模型响应成功
```python
["计算周一卖出的苹果数量：15个", "计算周二卖出的苹果数量：15 × 2 = 30个", "计算周三卖出的苹果数量：30 − 5 = 25个", "将三天卖出的苹果数量相加：15 + 30 + 25 = 70个"]
```
计划已生成: ```python
["计算周一卖出的苹果数量：15个", "计算周二卖出的苹果数量：15 × 2 = 30个", "计算周三卖出的苹果数量：30 − 5 = 25个", "将三天卖出的苹果数量相加：15 + 30 + 25 = 70个"]
```
生成的计划: ['计算周一卖出的苹果数量：15个', '计算周二卖出的苹果数量：15 × 2 = 30个', '计算周三卖出的苹果数量：30 − 5 = 25个', '将三天卖出的苹果数量相加：15 + 30 + 25 = 70个']
执行步骤1/4
正在调用qwen-plus模型...
大语言模型响应成功
15
步骤1结果: 15
执行步骤2/4
正在调用qwen-plus模型...
大语言模型响应成功
30
步骤2结果: 30
执行步骤3/4
正在调用qwen-plus模型...
大语言模型响应成功
25
步骤3结果: 25
执行步骤4/4
正在调用qwen-plus模型...
大语言模型响应成功
70
步骤4结果: 70
最终答案: 70


In [36]:
print("="*30)